In [16]:

# ============================================================
# Camada Gold - Análise da Alfabetização no Brasil
# ============================================================
# Objetivo:
# Construir tabelas analíticas finais a partir da camada Silver.
#
# Tabelas Gold geradas:
# 1. gold.indicador_meta_brasil
# 2. gold.indicador_meta_uf
# 3. gold.ranking_uf_prioritaria
# 4. gold.indicador_meta_municipio
# 5. gold.ranking_municipio_prioritario
# 6. gold.evolucao_alfabetizacao
# 7. gold.resumo_status_meta
#
# Também gera:
# docs/dicionario_dados_gold.md
#
# Observação:
# A Gold deve consumir a Silver. Como a Silver de meta municipal
# ainda pode não existir no projeto, este notebook inclui uma etapa
# de apoio para criá-la a partir da Bronze caso necessário.
# ============================================================

from pathlib import Path
from datetime import date, datetime
import pandas as pd
import gc


# ============================================================
# Configurações gerais
# ============================================================

BRONZE_PATH = Path("../data/bronze")
SILVER_PATH = Path("../data/silver")
GOLD_PATH = Path("../data/gold")
DOCS_PATH = Path("../docs")

EXECUTION_DATE = date.today().isoformat()

print("Bronze path:", BRONZE_PATH.resolve())
print("Silver path:", SILVER_PATH.resolve())
print("Gold path:", GOLD_PATH.resolve())
print("Docs path:", DOCS_PATH.resolve())
print("Execution date:", EXECUTION_DATE)


# ============================================================
# Funções utilitárias
# ============================================================

def localizar_parquet_mais_recente(caminho_tabela: Path) -> Path:
    arquivos = list(caminho_tabela.rglob("*.parquet"))

    if not arquivos:
        raise FileNotFoundError(f"Nenhum arquivo Parquet encontrado em: {caminho_tabela}")

    return max(arquivos, key=lambda arquivo: arquivo.stat().st_mtime)


def carregar_parquet_mais_recente(
    caminho_base: Path,
    nome_tabela: str,
    colunas: list[str] | None = None,
    nome_camada: str = "camada",
) -> pd.DataFrame:
    caminho_tabela = caminho_base / nome_tabela
    arquivo = localizar_parquet_mais_recente(caminho_tabela)

    df = pd.read_parquet(arquivo, columns=colunas)

    print(f"[OK] {nome_camada}.{nome_tabela} carregada")
    print(f"     Arquivo: {arquivo}")
    print(f"     Linhas: {len(df)} | Colunas: {len(df.columns)}")

    return df


def carregar_bronze(nome_tabela: str, colunas: list[str] | None = None) -> pd.DataFrame:
    return carregar_parquet_mais_recente(
        BRONZE_PATH,
        nome_tabela,
        colunas=colunas,
        nome_camada="bronze",
    )


def carregar_silver(nome_tabela: str, colunas: list[str] | None = None) -> pd.DataFrame:
    return carregar_parquet_mais_recente(
        SILVER_PATH,
        nome_tabela,
        colunas=colunas,
        nome_camada="silver",
    )


def salvar_particionado(
    df: pd.DataFrame,
    caminho_base: Path,
    nome_tabela: str,
    nome_camada: str,
) -> Path:
    output_dir = caminho_base / nome_tabela / f"execution_date={EXECUTION_DATE}"
    output_dir.mkdir(parents=True, exist_ok=True)

    output_file = output_dir / f"{nome_tabela}.parquet"

    df.to_parquet(output_file, index=False)

    print(f"[OK] {nome_camada}.{nome_tabela} salva em: {output_file}")
    print(f"     Linhas: {len(df)} | Colunas: {len(df.columns)}")

    return output_file


def salvar_silver(df: pd.DataFrame, nome_tabela: str) -> Path:
    return salvar_particionado(df, SILVER_PATH, nome_tabela, "silver")


def salvar_gold(df: pd.DataFrame, nome_tabela: str) -> Path:
    return salvar_particionado(df, GOLD_PATH, nome_tabela, "gold")


def padronizar_texto(valor):
    if pd.isna(valor):
        return valor

    return str(valor).strip()


def padronizar_codigo(valor):
    if pd.isna(valor):
        return valor

    return str(valor).strip()


def aplicar_status_meta(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df["distancia_meta"] = (
        df["taxa_alfabetizacao"]
        - df["meta_alfabetizacao"]
    )

    df["flag_meta_atingida"] = (
        df["distancia_meta"] >= 0
    )

    df.loc[
        df["taxa_alfabetizacao"].isna()
        | df["meta_alfabetizacao"].isna(),
        "flag_meta_atingida"
    ] = pd.NA

    df["status_meta"] = "Sem informação"

    df.loc[
        df["flag_meta_atingida"] == True,
        "status_meta"
    ] = "Meta atingida"

    df.loc[
        df["flag_meta_atingida"] == False,
        "status_meta"
    ] = "Abaixo da meta"

    return df


def conferir_tabela(df: pd.DataFrame, nome_tabela: str, linhas: int = 10):
    print("\n" + "-" * 80)
    print(f"Conferência: {nome_tabela}")
    print("-" * 80)
    print(f"Linhas: {len(df)}")
    print(f"Colunas: {len(df.columns)}")

    if "status_meta" in df.columns:
        print("\nDistribuição de status_meta:")
        display(
            df["status_meta"]
            .value_counts(dropna=False)
            .reset_index(name="quantidade")
            .rename(columns={"index": "status_meta"})
        )

    display(df.head(linhas))


# ============================================================
# Pré-requisito de apoio - Silver de Meta Alfabetização Município
# ============================================================
# Motivo:
# A Gold de município depende de metas municipais. Caso a Silver
# correspondente ainda não tenha sido criada, esta função cria:
#
# silver.fato_resultado_meta_municipio
# silver.fato_meta_anual_municipio
#
# Recomendação de evolução:
# Depois, essa etapa pode ser movida para o notebook da Silver,
# mantendo a Gold consumindo somente Silver.
# ============================================================

def garantir_silver_meta_municipio():
    existe_resultado = localizar_tabela_silver_existe("fato_resultado_meta_municipio")
    existe_meta = localizar_tabela_silver_existe("fato_meta_anual_municipio")

    if existe_resultado and existe_meta:
        print("[OK] Silver de meta município já existe.")
        return

    print("\n" + "=" * 80)
    print("Pré-requisito: criando Silver de meta_alfabetizacao_municipio")
    print("=" * 80)

    df_meta_municipio_bronze = carregar_bronze(
        "meta_alfabetizacao_municipio",
        [
            "ano",
            "id_municipio",
            "id_municipio_nome",
            "rede",
            "taxa_alfabetizacao",
            "meta_alfabetizacao_2024",
            "meta_alfabetizacao_2025",
            "meta_alfabetizacao_2026",
            "meta_alfabetizacao_2027",
            "meta_alfabetizacao_2028",
            "meta_alfabetizacao_2029",
            "meta_alfabetizacao_2030",
            "nivel_alfabetizacao",
            "percentual_participacao",
        ],
    )

    df_meta_municipio_base = df_meta_municipio_bronze.copy()

    df_meta_municipio_base["ano"] = df_meta_municipio_base["ano"].astype("Int64")
    df_meta_municipio_base["id_municipio"] = df_meta_municipio_base["id_municipio"].apply(padronizar_codigo)
    df_meta_municipio_base["id_municipio_nome"] = df_meta_municipio_base["id_municipio_nome"].apply(padronizar_texto)
    df_meta_municipio_base["rede"] = df_meta_municipio_base["rede"].apply(padronizar_texto)

    colunas_numericas = [
        "taxa_alfabetizacao",
        "meta_alfabetizacao_2024",
        "meta_alfabetizacao_2025",
        "meta_alfabetizacao_2026",
        "meta_alfabetizacao_2027",
        "meta_alfabetizacao_2028",
        "meta_alfabetizacao_2029",
        "meta_alfabetizacao_2030",
        "nivel_alfabetizacao",
        "percentual_participacao",
    ]

    for coluna in colunas_numericas:
        if coluna in df_meta_municipio_base.columns:
            df_meta_municipio_base[coluna] = pd.to_numeric(
                df_meta_municipio_base[coluna],
                errors="coerce"
            )

    df_meta_municipio_base = df_meta_municipio_base.drop_duplicates()

    df_fato_resultado_meta_municipio = df_meta_municipio_base[
        [
            "ano",
            "id_municipio",
            "rede",
            "taxa_alfabetizacao",
            "nivel_alfabetizacao",
            "percentual_participacao",
        ]
    ].copy()

    df_fato_resultado_meta_municipio["nivel_agregacao"] = "Município"
    df_fato_resultado_meta_municipio["data_processamento_silver"] = EXECUTION_DATE

    df_fato_resultado_meta_municipio = (
        df_fato_resultado_meta_municipio
        .drop_duplicates()
        .sort_values(["ano", "id_municipio", "rede"])
        .reset_index(drop=True)
    )

    colunas_metas = [
        "meta_alfabetizacao_2024",
        "meta_alfabetizacao_2025",
        "meta_alfabetizacao_2026",
        "meta_alfabetizacao_2027",
        "meta_alfabetizacao_2028",
        "meta_alfabetizacao_2029",
        "meta_alfabetizacao_2030",
    ]

    df_fato_meta_anual_municipio = df_meta_municipio_base.melt(
        id_vars=[
            "ano",
            "id_municipio",
            "rede",
        ],
        value_vars=colunas_metas,
        var_name="ano_meta",
        value_name="meta_alfabetizacao"
    )

    df_fato_meta_anual_municipio["ano_meta"] = (
        df_fato_meta_anual_municipio["ano_meta"]
        .str.extract(r"(\d{4})")
        .astype("Int64")
    )

    df_fato_meta_anual_municipio["nivel_agregacao"] = "Município"
    df_fato_meta_anual_municipio["data_processamento_silver"] = EXECUTION_DATE

    df_fato_meta_anual_municipio = (
        df_fato_meta_anual_municipio
        .drop_duplicates()
        .sort_values(["ano", "id_municipio", "rede", "ano_meta"])
        .reset_index(drop=True)
    )

    salvar_silver(df_fato_resultado_meta_municipio, "fato_resultado_meta_municipio")
    salvar_silver(df_fato_meta_anual_municipio, "fato_meta_anual_municipio")

    del df_meta_municipio_bronze
    del df_meta_municipio_base
    gc.collect()


def localizar_tabela_silver_existe(nome_tabela: str) -> bool:
    caminho_tabela = SILVER_PATH / nome_tabela

    if not caminho_tabela.exists():
        return False

    arquivos = list(caminho_tabela.rglob("*.parquet"))

    return len(arquivos) > 0


# ============================================================
# Dicionário Gold
# ============================================================

def gerar_dicionario_gold():
    DOCS_PATH.mkdir(parents=True, exist_ok=True)

    output_file = DOCS_PATH / "dicionario_dados_gold.md"

    descricoes_tabelas = {
        "indicador_meta_brasil": "Tabela analítica nacional que compara taxa de alfabetização observada com a meta do mesmo ano.",
        "indicador_meta_uf": "Tabela analítica por UF que compara taxa de alfabetização observada com a meta do mesmo ano.",
        "ranking_uf_prioritaria": "Ranking de UFs priorizadas conforme distância em relação à meta de alfabetização.",
        "indicador_meta_municipio": "Tabela analítica por município que compara taxa de alfabetização observada com a meta do mesmo ano.",
        "ranking_municipio_prioritario": "Ranking de municípios priorizados conforme distância em relação à meta de alfabetização.",
        "evolucao_alfabetizacao": "Tabela analítica consolidada para acompanhar a evolução da alfabetização por nível de agregação.",
        "resumo_status_meta": "Tabela consolidada com a quantidade e percentual de registros por status da meta, ano e nível de agregação.",
    }

    descricoes_colunas = {
        "ano": "Ano de referência do resultado observado.",
        "ano_meta": "Ano da meta comparada.",
        "rede": "Rede de ensino.",
        "nivel_agregacao": "Nível territorial da análise.",
        "sigla_uf": "Sigla da Unidade Federativa.",
        "sigla_uf_nome": "Nome da Unidade Federativa.",
        "id_municipio": "Código identificador do município.",
        "id_municipio_nome": "Nome do município.",
        "taxa_alfabetizacao": "Taxa de alfabetização observada.",
        "taxa_alfabetizacao_media": "Média da taxa de alfabetização observada no agrupamento.",
        "percentual_participacao": "Percentual de participação na avaliação.",
        "percentual_participacao_medio": "Média do percentual de participação no agrupamento.",
        "meta_alfabetizacao": "Meta de alfabetização prevista para o ano.",
        "meta_alfabetizacao_media": "Média da meta de alfabetização no agrupamento.",
        "distancia_meta": "Diferença entre taxa observada e meta de alfabetização.",
        "distancia_media_meta": "Média da distância entre taxa observada e meta de alfabetização.",
        "flag_meta_atingida": "Indica se a meta foi atingida.",
        "status_meta": "Classificação textual do atingimento da meta.",
        "posicao_prioridade": "Posição no ranking de priorização.",
        "total_registros": "Quantidade de registros considerados no agrupamento.",
        "total_meta_atingida": "Quantidade de registros com meta atingida.",
        "total_abaixo_meta": "Quantidade de registros abaixo da meta.",
        "quantidade": "Quantidade de registros no agrupamento.",
        "percentual_registros": "Percentual de registros no agrupamento.",
        "percentual_meta_atingida": "Percentual de registros com meta atingida.",
        "data_processamento_gold": "Data de processamento da camada Gold.",
    }

    tabelas_gold = []

    if GOLD_PATH.exists():
        for caminho_tabela in sorted(GOLD_PATH.iterdir()):
            if caminho_tabela.is_dir():
                try:
                    arquivo = localizar_parquet_mais_recente(caminho_tabela)
                    tabelas_gold.append(
                        {
                            "nome_tabela": caminho_tabela.name,
                            "arquivo": arquivo,
                        }
                    )
                except FileNotFoundError:
                    pass

    linhas_md = []

    linhas_md.append("# Dicionário de Dados - Camada Gold")
    linhas_md.append("")
    linhas_md.append(f"**Gerado em:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    linhas_md.append("")
    linhas_md.append("## Visão geral")
    linhas_md.append("")
    linhas_md.append(
        "A camada Gold contém tabelas analíticas finais derivadas da camada Silver, "
        "criadas para responder às perguntas de negócio relacionadas à alfabetização no Brasil."
    )
    linhas_md.append("")
    linhas_md.append(f"**Total de tabelas Gold identificadas:** {len(tabelas_gold)}")
    linhas_md.append("")

    linhas_md.append("## Tabelas identificadas")
    linhas_md.append("")
    linhas_md.append("| Tabela | Arquivo mais recente |")
    linhas_md.append("|---|---|")

    for item in tabelas_gold:
        linhas_md.append(
            f"| `gold.{item['nome_tabela']}` | `{item['arquivo'].as_posix()}` |"
        )

    linhas_md.append("")
    linhas_md.append("---")
    linhas_md.append("")

    for item in tabelas_gold:
        nome_tabela = item["nome_tabela"]
        arquivo = item["arquivo"]
        df = pd.read_parquet(arquivo)

        descricao_tabela = descricoes_tabelas.get(
            nome_tabela,
            "Tabela analítica da camada Gold."
        )

        linhas_md.append(f"## gold.{nome_tabela}")
        linhas_md.append("")
        linhas_md.append(f"**Descrição:** {descricao_tabela}")
        linhas_md.append("")
        linhas_md.append(f"**Arquivo físico:** `{arquivo.as_posix()}`")
        linhas_md.append("")
        linhas_md.append(f"**Quantidade de linhas:** {len(df)}")
        linhas_md.append("")
        linhas_md.append(f"**Quantidade de colunas:** {len(df.columns)}")
        linhas_md.append("")

        linhas_md.append("| Coluna | Tipo | Nulos | % Nulos | Valores distintos | Exemplo | Descrição |")
        linhas_md.append("|---|---|---:|---:|---:|---|---|")

        for coluna in df.columns:
            tipo = str(df[coluna].dtype)
            qtd_nulos = int(df[coluna].isna().sum())
            percentual_nulos = round(df[coluna].isna().mean() * 100, 2)
            qtd_distintos = int(df[coluna].nunique(dropna=True))

            valores_validos = df[coluna].dropna()
            exemplo = "" if valores_validos.empty else str(valores_validos.iloc[0])

            descricao_coluna = descricoes_colunas.get(
                coluna,
                "Campo gerado na camada Gold."
            )

            linhas_md.append(
                f"| `{coluna}` "
                f"| `{tipo}` "
                f"| {qtd_nulos} "
                f"| {percentual_nulos}% "
                f"| {qtd_distintos} "
                f"| `{exemplo}` "
                f"| {descricao_coluna} |"
            )

        linhas_md.append("")

    output_file.write_text("\n".join(linhas_md), encoding="utf-8")

    print("[OK] Dicionário Gold gerado")
    print(f"Arquivo: {output_file.resolve()}")


# ============================================================
# 1. gold.indicador_meta_brasil
# ============================================================

print("\n" + "=" * 80)
print("1. Gerando gold.indicador_meta_brasil")
print("=" * 80)

df_resultado_brasil = carregar_silver(
    "fato_resultado_brasil",
    [
        "ano",
        "rede",
        "taxa_alfabetizacao",
        "percentual_participacao",
        "nivel_agregacao",
    ]
)

df_meta_brasil = carregar_silver(
    "fato_meta_anual_brasil",
    [
        "ano",
        "rede",
        "ano_meta",
        "meta_alfabetizacao",
        "nivel_agregacao",
    ]
)

df_indicador_meta_brasil = df_resultado_brasil.merge(
    df_meta_brasil,
    on=[
        "ano",
        "rede",
        "nivel_agregacao",
    ],
    how="left"
)

df_indicador_meta_brasil = df_indicador_meta_brasil[
    df_indicador_meta_brasil["ano"] == df_indicador_meta_brasil["ano_meta"]
].copy()

df_indicador_meta_brasil = aplicar_status_meta(df_indicador_meta_brasil)

df_indicador_meta_brasil["data_processamento_gold"] = EXECUTION_DATE

df_indicador_meta_brasil = df_indicador_meta_brasil[
    [
        "ano",
        "rede",
        "nivel_agregacao",
        "taxa_alfabetizacao",
        "percentual_participacao",
        "ano_meta",
        "meta_alfabetizacao",
        "distancia_meta",
        "flag_meta_atingida",
        "status_meta",
        "data_processamento_gold",
    ]
].copy()

df_indicador_meta_brasil = (
    df_indicador_meta_brasil
    .sort_values(["ano", "rede"])
    .reset_index(drop=True)
)

conferir_tabela(df_indicador_meta_brasil, "gold.indicador_meta_brasil")
salvar_gold(df_indicador_meta_brasil, "indicador_meta_brasil")


# ============================================================
# 2. gold.indicador_meta_uf
# ============================================================

print("\n" + "=" * 80)
print("2. Gerando gold.indicador_meta_uf")
print("=" * 80)

df_resultado_meta_uf = carregar_silver(
    "fato_resultado_meta_uf",
    [
        "ano",
        "sigla_uf",
        "rede",
        "taxa_alfabetizacao",
        "percentual_participacao",
        "nivel_agregacao",
    ]
)

df_meta_anual_uf = carregar_silver(
    "fato_meta_anual_uf",
    [
        "ano",
        "sigla_uf",
        "rede",
        "ano_meta",
        "meta_alfabetizacao",
        "nivel_agregacao",
    ]
)

df_dim_uf = carregar_silver(
    "dim_uf",
    [
        "sigla_uf",
        "sigla_uf_nome",
    ]
)

df_indicador_meta_uf = df_resultado_meta_uf.merge(
    df_meta_anual_uf,
    on=[
        "ano",
        "sigla_uf",
        "rede",
        "nivel_agregacao",
    ],
    how="left"
)

df_indicador_meta_uf = df_indicador_meta_uf[
    df_indicador_meta_uf["ano"] == df_indicador_meta_uf["ano_meta"]
].copy()

df_indicador_meta_uf = df_indicador_meta_uf.merge(
    df_dim_uf,
    on="sigla_uf",
    how="left"
)

df_indicador_meta_uf = aplicar_status_meta(df_indicador_meta_uf)

df_indicador_meta_uf["data_processamento_gold"] = EXECUTION_DATE

df_indicador_meta_uf = df_indicador_meta_uf[
    [
        "ano",
        "sigla_uf",
        "sigla_uf_nome",
        "rede",
        "nivel_agregacao",
        "taxa_alfabetizacao",
        "percentual_participacao",
        "ano_meta",
        "meta_alfabetizacao",
        "distancia_meta",
        "flag_meta_atingida",
        "status_meta",
        "data_processamento_gold",
    ]
].copy()

df_indicador_meta_uf = (
    df_indicador_meta_uf
    .sort_values(["ano", "sigla_uf", "rede"])
    .reset_index(drop=True)
)

conferir_tabela(df_indicador_meta_uf, "gold.indicador_meta_uf")
salvar_gold(df_indicador_meta_uf, "indicador_meta_uf")


# ============================================================
# 3. gold.ranking_uf_prioritaria
# ============================================================

print("\n" + "=" * 80)
print("3. Gerando gold.ranking_uf_prioritaria")
print("=" * 80)

df_ranking_uf_prioritaria = df_indicador_meta_uf.copy()

df_ranking_uf_prioritaria = df_ranking_uf_prioritaria[
    df_ranking_uf_prioritaria["status_meta"] == "Abaixo da meta"
].copy()

df_ranking_uf_prioritaria = (
    df_ranking_uf_prioritaria
    .sort_values(["ano", "distancia_meta"], ascending=[True, True])
    .reset_index(drop=True)
)

df_ranking_uf_prioritaria["posicao_prioridade"] = (
    df_ranking_uf_prioritaria
    .groupby("ano")
    .cumcount()
    + 1
)

df_ranking_uf_prioritaria = df_ranking_uf_prioritaria[
    [
        "ano",
        "posicao_prioridade",
        "sigla_uf",
        "sigla_uf_nome",
        "rede",
        "taxa_alfabetizacao",
        "meta_alfabetizacao",
        "distancia_meta",
        "percentual_participacao",
        "status_meta",
        "data_processamento_gold",
    ]
].copy()

conferir_tabela(df_ranking_uf_prioritaria, "gold.ranking_uf_prioritaria")
salvar_gold(df_ranking_uf_prioritaria, "ranking_uf_prioritaria")


# ============================================================
# 4. gold.indicador_meta_municipio
# ============================================================

print("\n" + "=" * 80)
print("4. Gerando gold.indicador_meta_municipio")
print("=" * 80)

garantir_silver_meta_municipio()

df_resultado_meta_municipio = carregar_silver(
    "fato_resultado_meta_municipio",
    [
        "ano",
        "id_municipio",
        "rede",
        "taxa_alfabetizacao",
        "percentual_participacao",
        "nivel_agregacao",
    ]
)

df_meta_anual_municipio = carregar_silver(
    "fato_meta_anual_municipio",
    [
        "ano",
        "id_municipio",
        "rede",
        "ano_meta",
        "meta_alfabetizacao",
        "nivel_agregacao",
    ]
)

df_dim_municipio = carregar_silver(
    "dim_municipio",
    [
        "id_municipio",
        "id_municipio_nome",
    ]
)

df_indicador_meta_municipio = df_resultado_meta_municipio.merge(
    df_meta_anual_municipio,
    on=[
        "ano",
        "id_municipio",
        "rede",
        "nivel_agregacao",
    ],
    how="left"
)

df_indicador_meta_municipio = df_indicador_meta_municipio[
    df_indicador_meta_municipio["ano"] == df_indicador_meta_municipio["ano_meta"]
].copy()

df_indicador_meta_municipio = df_indicador_meta_municipio.merge(
    df_dim_municipio,
    on="id_municipio",
    how="left"
)

df_indicador_meta_municipio = aplicar_status_meta(df_indicador_meta_municipio)

df_indicador_meta_municipio["data_processamento_gold"] = EXECUTION_DATE

df_indicador_meta_municipio = df_indicador_meta_municipio[
    [
        "ano",
        "id_municipio",
        "id_municipio_nome",
        "rede",
        "nivel_agregacao",
        "taxa_alfabetizacao",
        "percentual_participacao",
        "ano_meta",
        "meta_alfabetizacao",
        "distancia_meta",
        "flag_meta_atingida",
        "status_meta",
        "data_processamento_gold",
    ]
].copy()

df_indicador_meta_municipio = (
    df_indicador_meta_municipio
    .sort_values(["ano", "id_municipio", "rede"])
    .reset_index(drop=True)
)

conferir_tabela(df_indicador_meta_municipio, "gold.indicador_meta_municipio")
salvar_gold(df_indicador_meta_municipio, "indicador_meta_municipio")


# ============================================================
# 5. gold.ranking_municipio_prioritario
# ============================================================

print("\n" + "=" * 80)
print("5. Gerando gold.ranking_municipio_prioritario")
print("=" * 80)

df_ranking_municipio_prioritario = df_indicador_meta_municipio.copy()

df_ranking_municipio_prioritario = df_ranking_municipio_prioritario[
    df_ranking_municipio_prioritario["status_meta"] == "Abaixo da meta"
].copy()

df_ranking_municipio_prioritario = (
    df_ranking_municipio_prioritario
    .sort_values(["ano", "distancia_meta"], ascending=[True, True])
    .reset_index(drop=True)
)

df_ranking_municipio_prioritario["posicao_prioridade"] = (
    df_ranking_municipio_prioritario
    .groupby("ano")
    .cumcount()
    + 1
)

df_ranking_municipio_prioritario = df_ranking_municipio_prioritario[
    [
        "ano",
        "posicao_prioridade",
        "id_municipio",
        "id_municipio_nome",
        "rede",
        "taxa_alfabetizacao",
        "meta_alfabetizacao",
        "distancia_meta",
        "percentual_participacao",
        "status_meta",
        "data_processamento_gold",
    ]
].copy()

conferir_tabela(df_ranking_municipio_prioritario, "gold.ranking_municipio_prioritario")
salvar_gold(df_ranking_municipio_prioritario, "ranking_municipio_prioritario")


# ============================================================
# 6. gold.evolucao_alfabetizacao
# ============================================================

print("\n" + "=" * 80)
print("6. Gerando gold.evolucao_alfabetizacao")
print("=" * 80)

df_evolucao_base = pd.concat(
    [
        df_indicador_meta_brasil,
        df_indicador_meta_uf,
        df_indicador_meta_municipio,
    ],
    ignore_index=True,
    sort=False
)

df_evolucao_alfabetizacao = (
    df_evolucao_base
    .groupby(["ano", "rede", "nivel_agregacao"], as_index=False)
    .agg(
        taxa_alfabetizacao_media=("taxa_alfabetizacao", "mean"),
        meta_alfabetizacao_media=("meta_alfabetizacao", "mean"),
        distancia_media_meta=("distancia_meta", "mean"),
        percentual_participacao_medio=("percentual_participacao", "mean"),
        total_registros=("status_meta", "count"),
        total_meta_atingida=("status_meta", lambda x: (x == "Meta atingida").sum()),
        total_abaixo_meta=("status_meta", lambda x: (x == "Abaixo da meta").sum()),
    )
)

df_evolucao_alfabetizacao["percentual_meta_atingida"] = (
    df_evolucao_alfabetizacao["total_meta_atingida"]
    / df_evolucao_alfabetizacao["total_registros"]
    * 100
).round(2)

df_evolucao_alfabetizacao["data_processamento_gold"] = EXECUTION_DATE

df_evolucao_alfabetizacao = (
    df_evolucao_alfabetizacao
    .sort_values(["nivel_agregacao", "ano", "rede"])
    .reset_index(drop=True)
)

conferir_tabela(df_evolucao_alfabetizacao, "gold.evolucao_alfabetizacao")
salvar_gold(df_evolucao_alfabetizacao, "evolucao_alfabetizacao")


# ============================================================
# 7. gold.resumo_status_meta
# ============================================================

print("\n" + "=" * 80)
print("7. Gerando gold.resumo_status_meta")
print("=" * 80)

df_resumo_status_meta = (
    df_evolucao_base
    .groupby(["ano", "rede", "nivel_agregacao", "status_meta"], as_index=False)
    .agg(quantidade=("status_meta", "count"))
)

df_total_status = (
    df_resumo_status_meta
    .groupby(["ano", "rede", "nivel_agregacao"], as_index=False)
    .agg(total_registros=("quantidade", "sum"))
)

df_resumo_status_meta = df_resumo_status_meta.merge(
    df_total_status,
    on=["ano", "rede", "nivel_agregacao"],
    how="left"
)

df_resumo_status_meta["percentual_registros"] = (
    df_resumo_status_meta["quantidade"]
    / df_resumo_status_meta["total_registros"]
    * 100
).round(2)

df_resumo_status_meta["data_processamento_gold"] = EXECUTION_DATE

df_resumo_status_meta = (
    df_resumo_status_meta
    .sort_values(["nivel_agregacao", "ano", "rede", "status_meta"])
    .reset_index(drop=True)
)

conferir_tabela(df_resumo_status_meta, "gold.resumo_status_meta")
salvar_gold(df_resumo_status_meta, "resumo_status_meta")


# ============================================================
# 8. Dicionário de dados Gold
# ============================================================

print("\n" + "=" * 80)
print("8. Gerando dicionário de dados Gold")
print("=" * 80)

gerar_dicionario_gold()


# ============================================================
# 9. Conferência final
# ============================================================

print("\n" + "=" * 80)
print("CONFERÊNCIA FINAL - CAMADA GOLD")
print("=" * 80)

tabelas_geradas = [
    "indicador_meta_brasil",
    "indicador_meta_uf",
    "ranking_uf_prioritaria",
    "indicador_meta_municipio",
    "ranking_municipio_prioritario",
    "evolucao_alfabetizacao",
    "resumo_status_meta",
]

for tabela in tabelas_geradas:
    arquivo = localizar_parquet_mais_recente(GOLD_PATH / tabela)
    df_conferencia = pd.read_parquet(arquivo)

    print(f"\ngold.{tabela}")
    print(f"Arquivo: {arquivo}")
    print(f"Dimensão: {df_conferencia.shape}")
    display(df_conferencia.head())

print("\nProcessamento Gold concluído com sucesso.")


Bronze path: C:\Projetos\fiap-tech-challenge-fase2\data\bronze
Silver path: C:\Projetos\fiap-tech-challenge-fase2\data\silver
Gold path: C:\Projetos\fiap-tech-challenge-fase2\data\gold
Docs path: C:\Projetos\fiap-tech-challenge-fase2\docs
Execution date: 2026-07-02

1. Gerando gold.indicador_meta_brasil
[OK] silver.fato_resultado_brasil carregada
     Arquivo: ..\data\silver\fato_resultado_brasil\execution_date=2026-06-30\fato_resultado_brasil.parquet
     Linhas: 3 | Colunas: 5
[OK] silver.fato_meta_anual_brasil carregada
     Arquivo: ..\data\silver\fato_meta_anual_brasil\execution_date=2026-06-30\fato_meta_anual_brasil.parquet
     Linhas: 21 | Colunas: 5

--------------------------------------------------------------------------------
Conferência: gold.indicador_meta_brasil
--------------------------------------------------------------------------------
Linhas: 2
Colunas: 11

Distribuição de status_meta:


C:\Users\Edy O Cruel\AppData\Local\Temp\ipykernel_22588\1216353490.py:151: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'nan' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  df.loc[


,status_meta,quantidade
0,Abaixo da meta,1
1,Meta atingida,1


,ano,rede,nivel_agregacao,taxa_alfabetizacao,percentual_participacao,ano_meta,meta_alfabetizacao,distancia_meta,flag_meta_atingida,status_meta,data_processamento_gold
0,2024,Pública,Brasil,59.2,87.37,2024,59.9,-0.7,False,Abaixo da meta,2026-07-02
1,2025,Pública,Brasil,66.0,88.00,2025,64.0,2.0,True,Meta atingida,2026-07-02


[OK] gold.indicador_meta_brasil salva em: ..\data\gold\indicador_meta_brasil\execution_date=2026-07-02\indicador_meta_brasil.parquet
     Linhas: 2 | Colunas: 11

2. Gerando gold.indicador_meta_uf
[OK] silver.fato_resultado_meta_uf carregada
     Arquivo: ..\data\silver\fato_resultado_meta_uf\execution_date=2026-06-30\fato_resultado_meta_uf.parquet
     Linhas: 81 | Colunas: 6
[OK] silver.fato_meta_anual_uf carregada
     Arquivo: ..\data\silver\fato_meta_anual_uf\execution_date=2026-06-30\fato_meta_anual_uf.parquet
     Linhas: 567 | Colunas: 6
[OK] silver.dim_uf carregada
     Arquivo: ..\data\silver\dim_uf\execution_date=2026-06-30\dim_uf.parquet
     Linhas: 25 | Colunas: 2

--------------------------------------------------------------------------------
Conferência: gold.indicador_meta_uf
--------------------------------------------------------------------------------
Linhas: 54
Colunas: 13

Distribuição de status_meta:


...


gold.evolucao_alfabetizacao
Arquivo: ..\data\gold\evolucao_alfabetizacao\execution_date=2026-07-02\evolucao_alfabetizacao.parquet
Dimensão: (5, 12)


,ano,rede,nivel_agregacao,taxa_alfabetizacao_media,meta_alfabetizacao_media,distancia_media_meta,percentual_participacao_medio,total_registros,total_meta_atingida,total_abaixo_meta,percentual_meta_atingida,data_processamento_gold
0,2024,Pública,Brasil,59.200000,59.900000,-0.700000,87.370000,1,0,1,0.00,2026-07-02
1,2025,Pública,Brasil,66.000000,64.000000,2.000000,88.000000,1,1,0,100.00,2026-07-02
2,2024,Municipal,Município,63.041747,62.151745,1.114841,91.027399,5352,2788,2444,52.09,2026-07-02
3,2024,Pública,UF,56.708846,58.233333,-1.403333,87.243077,27,11,13,40.74,2026-07-02
4,2025,Pública,UF,65.518519,62.230769,3.615385,88.259259,27,20,6,74.07,2026-07-02



gold.resumo_status_meta
Arquivo: ..\data\gold\resumo_status_meta\execution_date=2026-07-02\resumo_status_meta.parquet
Dimensão: (11, 8)


,ano,rede,nivel_agregacao,status_meta,quantidade,total_registros,percentual_registros,data_processamento_gold
0,2024,Pública,Brasil,Abaixo da meta,1,1,100.00,2026-07-02
1,2025,Pública,Brasil,Meta atingida,1,1,100.00,2026-07-02
2,2024,Municipal,Município,Abaixo da meta,2444,5352,45.67,2026-07-02
3,2024,Municipal,Município,Meta atingida,2788,5352,52.09,2026-07-02
4,2024,Municipal,Município,Sem informação,120,5352,2.24,2026-07-02



Processamento Gold concluído com sucesso.
